# 从零开始运行 GraphCast （AutoDL 或者其他新的环境）
-------------------------------------------------------------------
**这是从 https://google-deepmind/graphcast 复现的项目。由 https://github.com/sfsun67 改写和调试。**

**AutoDL 是国内的一家云计算平台，网址是https://www.autodl.com**

你应该有类似的文件结构，这里的数据由 Google Cloud Bucket (https://console.cloud.google.com/storage/browser/dm_graphcast 提供。模型权重、标准化统计和示例输入可在Google Cloud Bucket上找到。完整的模型训练需要下载ERA5数据集，该数据集可从ECMWF获得。
```
.
├── code
│   ├── GraphCast-from-Ground-Zero
│       ├──graphcast
│       ├──tree
│       ├──wrapt
│       ├──graphcast_demo.ipynb
│       ├──README.md
│       ├──setup.py
│       ├──...
├── data
│   ├── dataset
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-04.nc
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-12.nc
│       ├──...
│   ├── params
│       ├──params-GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
│       ├──params-GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
│       ├──...
│   ├── stats
│       ├──stats-mean_by_level.nc
│       ├──...
└────── 
```

PS: 
1. Python 要使用3.10版本。老版本会出现函数调用失效的问题。
2. 你需要仔细核对包的版本，防止出现意外的错误。例如， xarray 只能使用 2023.7.0 版本，其他版本会出现错误。
3. 你需要仔细核对所有包是否安装正确。未安装的包会导致意外错误。例如，tree 和 wrapt 是两个 GraphCast 所必需的包，但是并不在源文件中。例如，tree 和 wrapt 中的 .os 文件未导入，会引发循环调用。他们的原始文件可以在 Colaboratory(https://colab.research.google.com/github/deepmind/graphcast/blob/master/graphcast_demo.ipynb) 的环境中找到。



*代码在如下机器上测试*
1. GPU: TITAN Xp 12GB; CPU: Xeon(R) E5-2680 v4;  JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
2. GPU: V100-SXM2-32GB 32GB; CPU: Xeon(R) Platinum 8255C; JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
3. GPU: RTX 2080 Ti(11GB); CPU: Xeon(R) Platinum 8255C; JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
-------------------------------------------------------------------


<p><small><small>版权所有 2023 年 DeepMind Technologies Limited。</small></small></p>
<p><small><small>根据 Apache 许可证第 2.0 版（"许可证"）获得许可；除非符合许可证的规定，否则您不得使用此文件。您可以在 <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a> 获取许可证的副本。</small></small></p>
<p><small><small>除非适用法律要求或书面同意，根据许可证分发的软件是基于 "按原样" 分发的，没有任何明示或暗示的担保或条件。有关许可证下的具体语言，请参见许可证中的权限和限制。</small></small></p>


# 将 Python 版本更新到 3.10.

GraphCast 需要 Python >= 3.10 。推荐 Python 3.10。

在终端中，新建一个名为 GraphCast 的环境。

参考代码如下：
```

# 更新 conda （可选）
conda update -n base -c defaults conda

# 在新环境 GraphCast 中安装 python=3.10  
conda create -n GraphCast python=3.10    

# 更新bashrc中的环境变量
conda init bash && source /root/.bashrc

# 激活新的环境
conda activate GraphCast

# 验证版本
python --version

# 在 Jupyter 中注册 Python 3.10 环境
# 安装 ipykernel 包
conda install ipykernel

# 注册的 Python 3.10 环境的内核名称
python -m ipykernel install --user --name=GraphCast-python3.10
```

注意：Jupyter 注册 Python 3.10 环境后，重启jupyter，使用新的内核 GraphCast-python3.10。

# 安装和初始化


In [19]:
# # 学术资源加速 https://www.autodl.com/docs/network_turbo/  .

# import subprocess
# import os

# result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
# output = result.stdout
# for line in output.splitlines():
#     if '=' in line:
#         var, value = line.split('=', 1)
#         os.environ[var] = value

In [20]:
# # 这一步将使用 shapely 安装环境。为了避免出现ERROR： 无法为 shapely 构建轮子，而安装基于 pyproject.toml 的项目需要轮子。

# !pip uninstall -y shapely
# !conda install -y shapely
# !pip uninstall -y shapely

/root/miniconda3/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Found existing installation: shapely 2.0.4
Uninstalling shapely-2.0.4:
  Successfully uninstalled shapely-2.0.4
Retrieving notices: ...working... done
Solving environment: done

## Package Plan ##

  environment location: /root/miniconda3

  added / updated specs:
    - shapely


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2024.3.11  |       h06a4308_0         127 KB  https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
    certifi-2024.7.4           |  py310h06a4308_0         158 KB  https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
    ------------------------------------------------------------
                                           Total:         286 KB

The following packages will be UPDATED:

  certifi            conda-forge/noarch::certifi-2024.6.2-~ --> anaconda/pkgs/main/linux-64::certifi-2024.7.4-py310h06a4308_0 

The following packages will be 

In [21]:
# # @title Pip 安装 graphcast 和其他依赖项


# %pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
  Using cached https://github.com/deepmind/graphcast/archive/master.zip
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 168.0 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.8 kB/s eta 0:00:0000:0100:01m
     ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/79.6 MB 156.9 kB/s eta 0:07:23^C
     ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/79.6 MB 156.9 kB/s eta 0:07:23
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [22]:
# !pip install xarray

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [ ]:
# # @title cartopy 崩溃的解决方法

# !pip uninstall -y shapely
# !pip install shapely --no-binary shapely

DEPRECATION: --no-binary currently disables reading from the cache of locally built wheels. In the future --no-binary will not influence the wheel cache. pip 23.1 will enforce this behaviour change. A possible replacement is to use the --no-cache-dir option. You can use the flag --use-feature=no-binary-enable-wheel-cache to test the upcoming behaviour. Discussion can be found at https://github.com/pypa/pip/issues/11453
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
  Using cached http://mirrors.aliyun.com/pypi/packages/49/7e/816fd1c135b062c80b72e17b7330d9a719cd413158afa580f4aaccf59aa9/shapely-2.0.4.tar.gz (280 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for shapely: filename=shapely-2.0.4-cp310-cp310-linux_x86_64.whl size=416452 sha256=c281dad2a94b661cfd4fcec7b689335be60763d1db750ef4287192b15993bb4c
  Stored in directory: /root/.cache/pip/wheels/d4/2a/6b/e5f8e7d2e2f

In [ ]:
# @title 安装其他依赖项，并解决 xarray 的版本问题。

# 这里需要将xarray的版本从2023.12.0(2023年12月30日安装)降低到2023.7.0，否则会报错。

!conda install -y -c conda-forge ipywidgets
# !pip uninstall -y xarray
# !pip install xarray==2023.7.0

In [ ]:
# pip install ipywidgets

In [22]:
# !pip uninstall -y xarray
# !pip install xarray==2023.7.0

/root/miniconda3/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Found existing installation: xarray 2024.6.0
Uninstalling xarray-2024.6.0:
  Successfully uninstalled xarray-2024.6.0
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 116.5 kB/s eta 0:00:0000:0100:01


In [1]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray




def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [2]:
# @title 载入绘图函数

# 这定义了一个名为 select 的函数，它接受一个 xarray.Dataset 对象、一个指定数据集中变量的字符串，以及可选的水平和最大时间步数参数。它返回一个 xarray.Dataset。
def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
#     从数据集中选择与指定变量相对应的数据。
  data = data[variable]
#     如果数据集有一个名为 “batch” 的维度，这行代码选择数据的第一批。
  if "batch" in data.dims:
    data = data.isel(batch=0)
#     如果指定了 max_steps，并且数据集有一个 “time” 维度且步数多于 max_steps，这行代码将数据集限制在前 max_steps 个时间步。
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
#     如果指定了 level 并且它存在于数据集的坐标中，这行代码选择在该特定水平上的数据。
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

# 这定义了一个名为 scale 的函数，它接受一个 xarray.Dataset、一个用于缩放的可选中心值和一个表示是否使用鲁棒缩放的布尔值。
# 它返回一个包含数据集、matplotlib.colors.Normalize 对象和颜色映射名称的元组。
def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
#     这些行计算用于归一化的最小和最大值。如果 robust 为 True，它使用第2和第98百分位数来忽略异常值；否则，它使用绝对最小和最大值。
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
# 如果提供了 center 值，这些行调整 vmin 和 vmax 使其与中心等距，确保中心值是颜色刻度的中点。
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
#     函数返回数据集、配置有 vmin 和 vmax 的 Normalize 对象，以及要使用的颜色映射名称（如果有中心点则为 “RdBu_r”，否则为 “viridis”）。
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

# 这定义了一个名为 plot_data 的函数，它接受一个标题和 xarray.Dataset 对象的字典、图形标题、可选的绘图大小、鲁棒缩放标志和子图布局的列数。
# 它返回一个类似于 scale 函数的元组。
def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

# 这些行获取字典中的第一个数据集以确定时间步数（max_steps），并断言所有数据集都有相同数量的时间步。
  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

#     列数设置为指定列数或数据字典长度的最小值。根据列数计算行数。
  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
# 创建一个新的图形，其大小基于行数和列数以及指定的绘图大小。
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
#     设置图形的标题，并调整布局以消除子图之间的空白。
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

#     这个循环为字典中的每个数据集创建子图，设置轴和标题。
  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
#     每个子图使用 imshow 函数显示数据集的第一个时间步，使用指定的归一化和颜色映射。
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
#     为每个子图添加一个颜色条。
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

#     这定义了一个用于动画的 update 函数，它会在每个帧更新标题和数据。
  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

#     使用 FuncAnimation 类创建一个动画，它将为每个帧调用 update 函数。
  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
#     关闭图形（以防止它立即显示），并将动画转换为使用 JavaScript 的 HTML5 视频，然后返回。
  plt.close(figure.number)
  return HTML(ani.to_jshtml())

# 加载数据并初始化模型

## 载入模型参数

选择两种获取模型参数的方式之一：
- **random**：您将获得随机预测，但您可以更改模型架构，这可能会使其运行更快或适应您的设备。
- **checkpoint**：您将获得明智的预测，但受限于模型训练时使用的架构，这可能不适合您的设备。特别是生成梯度会使用大量内存，因此您至少需要25GB的内存（TPUv4或A100）。

检查点在一些方面有所不同：
- 网格大小指定了地球的内部图形表示。较小的网格将运行更快，但输出将更差。网格大小不影响模型的参数数量。
- 分辨率和压力级别的数量必须匹配数据。较低的分辨率和较少的级别会运行得更快。数据分辨率仅影响编码器/解码器。
- 我们的所有模型都预测降水。然而，ERA5包含降水，而HRES不包含。我们标记为 "ERA5" 的模型将降水作为输入，并期望以ERA5数据作为输入，而标记为 "ERA5-HRES" 的模型不以降水作为输入，并专门训练以HRES-fc0作为输入（请参阅下面的数据部分）。

我们提供三个预训练模型：
1. `GraphCast`，用于GraphCast论文的高分辨率模型（0.25度分辨率，37个压力级别），在1979年至2017年间使用ERA5数据进行训练，

2. `GraphCast_small`，GraphCast的较小低分辨率版本（1度分辨率，13个压力级别和较小的网格），在1979年至2015年间使用ERA5数据进行训练，适用于具有较低内存和计算约束的模型运行，

3. `GraphCast_operational`，一个高分辨率模型（0.25度分辨率，13个压力级别），在1979年至2017年使用ERA5数据进行预训练，并在2016年至2021年间使用HRES数据进行微调。此模型可以从HRES数据初始化（不需要降水输入）。


In [3]:
# @title 选择模型
# Rewrite by S.F. Sune, https://github.com/sfsun67.
'''
    我们有三种训练好的模型可供选择, 需要从https://console.cloud.google.com/storage/browser/dm_graphcast准备：
    GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
    GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz
    GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
'''
# 在此路径 /root/data/params 中查找结果，并列出 "params/"中所有文件的名称，去掉名称中的 "params/"perfix。

import os
import glob

# 定义数据目录，请替换成你自己的目录。
dir_path_params = "/root/data/params"


# Use glob to get all file paths in the directory
# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
file_paths_params = glob.glob(os.path.join(dir_path_params, "*"))

# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
# Remove the directory path and the ".../params/" prefix from each file name
params_file_options = [os.path.basename(path) for path in file_paths_params]

# 创建一个整数滑块控件random_mesh_size，用于选择网格大小。其默认值为4，最小值为4，最大值为6。
random_mesh_size = widgets.IntSlider(
    value=4, min=4, max=6, description="Mesh size:")
# 创建一个整数滑块控件random_gnn_msg_steps，用于选择GNN消息传递的步数。其默认值为4，最小值为1，最大值为32。
random_gnn_msg_steps = widgets.IntSlider(
    value=4, min=1, max=32, description="GNN message steps:")
# 创建一个下拉菜单控件random_latent_size，用于选择潜在大小。选项是2的4次方到2的9次方，即16到512，其默认值为32。
random_latent_size = widgets.Dropdown(
    options=[int(2**i) for i in range(4, 10)], value=32,description="Latent size:")
# 创建一个下拉菜单控件random_levels，用于选择压力水平。选项有13和37，其默认值为13。
random_levels = widgets.Dropdown(
    options=[7, 37], value=7, description="Pressure levels:")

# 创建一个下拉菜单控件params_file，用于选择参数文件。选项是之前从目录中获取的文件名列表。
params_file = widgets.Dropdown(
    options=params_file_options,
    description="Params file:",
    layout={"width": "max-content"})

# 创建一个标签页控件source_tab，其中包含两个标签。第一个标签是一个垂直布局的盒子，包含了前面创建的四个控件。第二个标签是参数文件的下拉菜单。
source_tab = widgets.Tab([
    widgets.VBox([
        random_mesh_size,
        random_gnn_msg_steps,
        random_latent_size,
        random_levels,
    ]),
    params_file,
])
# 为source_tab的两个标签设置标题，分别为"随机参数权重"和"预训练权重"。
source_tab.set_title(0, "随机参数权重（Random）")
source_tab.set_title(1, "预训练权重（Checkpoint）")
# 最后，创建一个垂直布局的盒子，包含了source_tab和一个标签，提示用户运行下一个单元格以加载模型，并告知重新运行该单元格将清除他们的选择。
widgets.VBox([
    source_tab,
    widgets.Label(value="运行下一个单元格以加载模型。重新运行该单元格将清除您的选择。")
])


In [4]:
# @title 加载模型

# 这行代码获取当前选中的标签页的标题，以确定用户选择了哪种参数权重（随机参数权重或预训练权重）。
source = source_tab.get_title(source_tab.selected_index)

# 如果用户选择了“随机参数权重”，则执行以下代码块。
if source == "随机参数权重（Random）":
#     初始化params为None和state为一个空字典。这些将在后面的代码中使用
  params = None  # Filled in below
  state = {}
# 创建一个model_config对象，它包含模型配置的参数。这些参数包括：
# resolution：分辨率，这里设置为0。
# mesh_size：网格大小，取自之前创建的滑块控件的值。
# latent_size：潜在大小，取自下拉菜单控件的值。
# gnn_msg_steps：GNN消息传递步数，取自滑块控件的值。
# hidden_layers：隐藏层的数量，这里设置为1。
# radius_query_fraction_edge_length：查询半径与边长的比例，这里设置为0.6。
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.6)
#     创建一个task_config对象，它包含任务配置的参数。这些参数包括：
# input_variables：输入变量。
# target_variables：目标变量。
# forcing_variables：强迫变量。
# pressure_levels：压力水平，取自下拉菜单控件的值。
# input_duration：输入持续时间。
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
#     如果用户选择了“预训练权重”，则执行以下代码块。
# 这段被注释的代码原本用于从Google Cloud Storage加载预训练权重。
# 现在，它被替换为从本地文件系统加载预训练权重的代码。使用open函数以二进制读取模式打开参数文件，并使用checkpoint.load函数加载检查点。
else:
  assert source == "预训练权重（Checkpoint）"
  '''with gcs_bucket.blob(f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)'''
  
  with open(f"{dir_path_params}/{params_file.value}", "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
    
#  从检查点中提取参数到params变量，并重新初始化state为一个空字典。 
  params = ckpt.params
  state = {}

# 从检查点中提取模型配置和任务配置。
  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("模型描述:\n", ckpt.description, "\n")
  print("模型许可信息:\n", ckpt.license, "\n")

model_config

ModelConfig(resolution=0, mesh_size=4, latent_size=32, gnn_msg_steps=4, hidden_layers=1, radius_query_fraction_edge_length=0.6, mesh2grid_edge_normalization_factor=None)



## 载入示例数据

有几个示例数据集可用，在几个坐标轴上各不相同：
- **来源**：fake、era5、hres
- **分辨率**：0.25度、1度、6度
- **级别**：13, 37
- **步数**：包含多少个时间步

并非所有组合都可用。
- 由于加载内存的要求，较高分辨率只适用于较少的步数。
- HRES 只有 0.25 度，13 个压力等级。

数据分辨率必须与加载的模型相匹配。

对基础数据集进行了一些转换：
- 我们累积了 6 个小时的降水量，而不是默认的 1 个小时。
- 对于 HRES 数据，每个时间步对应 HRES 在前导时间 0 的预报，实际上提供了 HRES 的 "初始化"。有关详细描述，请参见 GraphCast 论文中的 HRES-fc0。请注意，HRES 无法提供 6 小时的累积降水量，因此我们的模型以 HRES 输入不依赖于降水。但由于我们的模型可以预测降水，因此在示例数据中包含了 ERA5 降水量，以作为地面真实情况的示例。
- 我们在数据中加入了 ERA5 的 "toa_incident_solar_radiation"。我们的模型使用 -6h、0h 和 +6h 辐射作为每 1 步预测的强迫项。在运行中，如果没有现成的 +6h 辐射，可以使用诸如 `pysolar` 等软件包计算辐射。


In [5]:
# @title 获取和筛选可用示例数据的列表

# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 在“/root/data/dataset”路径下查找结果，并列出“dataset/”中所有文件的名称列表，去掉“dataset/”前缀。

# # 定义数据目录，请替换成你自己的目录。
# dir_path_dataset = "rood/code/GraphCast-from-Ground-Zero/"

# # Use glob to get all file paths in the directory
# file_paths_dataset = glob.glob(os.path.join(dir_path_dataset, "*"))

# # Remove the directory path and the ".../params/" prefix from each file name
# dataset_file_options = [os.path.basename(path) for path in file_paths_dataset]
# #print("dataset_file_options: ", dataset_file_options)

# # Remove "dataset-" prefix from each file name
# dataset_file_options = [name.removeprefix("GraphCast-from-Ground-Zero-") for name in dataset_file_options]

# # 定义了一个名为data_valid_for_model的函数，它接受三个参数：file_name（数据文件名），model_config（模型配置对象），和task_config（任务配置对象）。
# def data_valid_for_model(
#     file_name: str, model_config: graphcast.ModelConfig, task_config: graphcast.TaskConfig):
# #     调用parse_file_parts函数来解析文件名，移除文件扩展名.nc。假设parse_file_parts是一个自定义函数，用于将文件名分解为不同的部分。
#   file_parts = parse_file_parts(file_name.removesuffix(".nc"))
#   # print("file_parts: ", file_parts)

# # 这个函数返回一个布尔值，表示文件是否适合模型配置。它检查以下条件：
# # 模型配置的分辨率是否为0或与文件部分中的res值相匹配。
# # 任务配置的压力水平数量是否与文件部分中的levels值相匹配。
# # 如果任务配置的输入变量中包含total_precipitation_6hr，则文件来源必须是era5或fake；如果不包含，则文件来源必须是hres或fake。
#   return (
#       model_config.resolution in (0, float(file_parts["res"])) and
#       len(task_config.pressure_levels) == int(file_parts["levels"]) and
#       (
#           ("total_precipitation_6hr" in task_config.input_variables and
#            file_parts["source"] in ("era5", "fake")) or
#           ("total_precipitation_6hr" not in task_config.input_variables and
#            file_parts["source"] in ("hres", "fake"))
#       )
#   )

# # 创建一个下拉菜单控件dataset_file，它的选项是通过列表推导式生成的，只包括那些通过data_valid_for_model函数验证为有效的数据文件。
# # 每个选项都是一个元组，第一个元素是文件部分的字符串表示，第二个元素是原始文件名。
# dataset_file = widgets.Dropdown(
#     options=[
#         (", ".join([f"{k}: {v}" for k, v in parse_file_parts(option.removesuffix(".nc")).items()]), option)
#         for option in dataset_file_options
#         if data_valid_for_model(option, model_config, task_config)
#     ],
#     description="数据文件:",
#     layout={"width": "max-content"})
# # 创建一个垂直盒子控件，包含了dataset_file下拉菜单和一个标签，提示用户运行下一个单元格以加载数据集，并告知重新运行该单元格将清除他们的选择并重新筛选数据集。
# widgets.VBox([
#     dataset_file,
#     widgets.Label(value="运行下一个单元格以加载数据集。重新运行此单元格将清除您的选择，并重新筛选与您的模型匹配的数据集。")
# ])

In [6]:
# import xarray as xr
# # 假设data是您已经加载的数据集
# data = xr.open_dataset('dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-12.nc')

# # 直接打印batch维度的数据类型
# print(data.coords['batch'].dtype)
# # 打印batch维度的原始数据
# print(data.coords['batch'].values)# 加载数据集
# file_path = "dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-12.nc"
# with open(file_path, "rb") as f:
#     example_batch = xr.load_dataset(f).compute()

# # 假设'time'维度已经是datetime64[ns]类型
# # 并且'batch'维度长度为1

# # 如果datetime是坐标，直接打印它
# print("Datetime Coordinates:")
# print(example_batch.coords['datetime'])

In [7]:
# import xarray as xr
# import pandas as pd
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"

# with open(f"{dir_path_data}/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_180.00W-179.92E_80.00S-90.00N_0.49-7.93m_2021-04-30-2021-06-30.nc", "rb") as f:
#   example_batch = xarray.load_dataset(f).compute()
# # 重命名维度
# example_batch = example_batch.rename({
#     'latitude': 'lat',
#     'longitude': 'lon',
#     'depth': 'level',
# })

# # 定义一个重新采样函数
# def coarsen_data(ds, factor):
#     coarsened = ds.coarsen(lat=factor, lon=factor, boundary='trim').mean()
#     return coarsened

# # 设置降采样因子
# coarsen_factor = 4
# example_batch = coarsen_data(example_batch, coarsen_factor)

# # 获取原始纬度
# original_lat = example_batch.lat.values
# delta_latitude = np.abs(original_lat[1] - original_lat[0])
# new_lat = np.arange(-90 + delta_latitude/2, 90 - delta_latitude/2, delta_latitude)
# example_batch = example_batch.interp(lat=new_lat)

In [8]:
import xarray as xr
import pandas as pd
import numpy as np

dir_path_data = "/root/autodl-tmp/"

with open(f"{dir_path_data}/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_180.00W-179.92E_80.00S-90.00N_0.49-7.93m_2021-04-30-2021-06-30.nc", "rb") as f:
  example_batch = xarray.load_dataset(f).compute()

# 重命名维度
example_batch = example_batch.rename({
    'latitude': 'lat',
    'longitude': 'lon',
    'depth': 'level',
})

# 定义一个重新采样函数
def coarsen_data(ds, factor):
    coarsened = ds.coarsen(lat=factor, lon=factor, boundary='trim').mean()
    return coarsened

# 设置降采样因子
coarsen_factor = 4
example_batch = coarsen_data(example_batch, coarsen_factor)

# 获取原始纬度
original_lat = example_batch.lat.values
delta_latitude = np.abs(original_lat[1] - original_lat[0])
new_lat = np.arange(-90 + delta_latitude/2, 90 - delta_latitude/2, delta_latitude)
example_batch = example_batch.interp(lat=new_lat)

# 获取现有的时间坐标
time_coords = example_batch.coords['time'].values

# 计算新的batch维度大小
num_days = example_batch.dims['time']
batch_size = 7
num_batches = (num_days + batch_size - 1) // batch_size  # 向上取整以确保包含所有天数

# 创建一个新的时间坐标数组，大小为 (num_batches, batch_size)
datetime_coords = np.full((num_batches, batch_size), np.datetime64('NaT'), dtype='datetime64[ns]')
for i in range(num_batches):
    start = i * batch_size
    end = min((i + 1) * batch_size, num_days)
    datetime_coords[i, :end-start] = time_coords[start:end]

# 创建新的数据变量字典，重塑数据
new_data_vars = {}
for var in example_batch.data_vars:
    data = example_batch[var].values
    reshaped_data = np.empty((num_batches, batch_size, *data.shape[1:]), dtype=data.dtype)
    for i in range(num_batches):
        start = i * batch_size
        end = min((i + 1) * batch_size, num_days)
        reshaped_data[i, :end-start] = data[start:end]
    new_data_vars[var] = (['batch', 'time'] + list(example_batch[var].dims[1:]), reshaped_data)

# 创建新的数据集，添加 batch 和 time 维度
example_batch = xr.Dataset(new_data_vars, coords={
    'level': example_batch['level'],
    'lon': example_batch['lon'],
    'lat': example_batch['lat'],
    'time': ('time', np.arange(batch_size)),  # 使用0到batch_size-1的整数作为新的时间坐标
    'batch': ('batch', np.arange(num_batches)),
    'datetime': (['batch', 'time'], datetime_coords),
})

# 恢复 time 维度为 timedelta64[ns]
example_batch['time'] = pd.to_timedelta(example_batch['time'], unit='D')

# 显示修改后的数据集
example_batch

<xarray.Dataset>
Dimensions:   (batch: 9, time: 7, lat: 539, lon: 1080, level: 7)
Coordinates:
  * level     (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon       (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat       (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 5 days 6 days
  * batch     (batch) int64 0 1 2 3 4 5 6 7 8
    datetime  (batch, time) datetime64[ns] 2021-04-30 2021-05-01 ... NaT
Data variables:
    siconc    (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    sithick   (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    so        (batch, time, level, lat, lon) float64 nan nan nan ... 0.0 0.0 0.0
    thetao    (batch, time, level, lat, lon) float64 nan nan nan ... 0.0 0.0 0.0
    zos       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    usi       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    vsi       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0

In [9]:
import xarray as xr
import numpy as np

# 常量
SEC_PER_DAY = 86400
_AVG_DAY_PER_YEAR = 365.25

# 计算进度
seconds_since_epoch = (example_batch['time'].astype('int64') // 10**9).values
longitude = example_batch['lon'].values  # 经度

year_progress = data_utils.get_year_progress(seconds_since_epoch)
day_progress = data_utils.get_day_progress(seconds_since_epoch, longitude)

# 生成进度特征
year_features = data_utils.featurize_progress('year_progress', ['time'], year_progress)
day_features = data_utils.featurize_progress('day_progress', ['time', 'lon'], day_progress)

example_batch = example_batch.assign(year_progress_sin=year_features['year_progress_sin'],
                               year_progress_cos=year_features['year_progress_cos'],
                               day_progress_sin=day_features['day_progress_sin'],
                               day_progress_cos=day_features['day_progress_cos'])

# 计算每一天的全图均值和标准差
daily_mean = example_batch.mean(dim=['lat', 'lon','time','batch'], skipna=True)
daily_stddev = example_batch.std(dim=['lat', 'lon','time','batch'], skipna=True)

# 将数据标准化
normalized_by_level = (example_batch - daily_mean) / daily_stddev

# 只保留level维度，计算标准化后的均值
normalized_by_level_mean = normalized_by_level.mean(dim=['lat', 'lon', 'time', 'batch'], skipna=True)



# 检查目标目录是否存在并具有写入权限
import os
save_path = '/root/data/stats'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 分别保存均值、标准差和标准化值到三个独立的NetCDF文件
try:
    daily_mean.to_netcdf(f'{save_path}/stats-mean_by_level.nc', engine='scipy')
    print("均值文件已保存")
except RuntimeError as e:
    print(f"保存均值文件时出错: {e}")

try:
    daily_stddev.to_netcdf(f'{save_path}/stats-stddev_by_level.nc', engine='scipy')
    print("标准差文件已保存")
except RuntimeError as e:
    print(f"保存标准差文件时出错: {e}")

try:
    normalized_by_level_mean.to_netcdf(f'{save_path}/stats-diffs_stddev_by_level.nc', engine='scipy')
    print("标准化文件已保存")
except RuntimeError as e:
    print(f"保存标准化文件时出错: {e}")

print("文件保存完成")


{1}
{1}
{2}
{2}
均值文件已保存
标准差文件已保存
标准化文件已保存
文件保存完成


In [10]:
# 去除进度特征
example_batch = example_batch.drop_vars([
    'year_progress_sin', 'year_progress_cos',
    'day_progress_sin', 'day_progress_cos'
])
example_batch

<xarray.Dataset>
Dimensions:   (batch: 9, time: 7, lat: 539, lon: 1080, level: 7)
Coordinates:
  * level     (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon       (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat       (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 5 days 6 days
  * batch     (batch) int64 0 1 2 3 4 5 6 7 8
    datetime  (batch, time) datetime64[ns] 2021-04-30 2021-05-01 ... NaT
Data variables:
    siconc    (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    sithick   (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    so        (batch, time, level, lat, lon) float64 nan nan nan ... 0.0 0.0 0.0
    thetao    (batch, time, level, lat, lon) float64 nan nan nan ... 0.0 0.0 0.0
    zos       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    usi       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0
    vsi       (batch, time, lat, lon) float64 nan nan nan nan ... 0.0 0.0 0.0

In [11]:
# @title 加载规范化数据
# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
import xarray as xr
import os

dir_path_stats = "/root/data/stats/"

# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()
# 类似于第4行，这行代码打开了另一个文件stats-mean_by_level.nc。
with open(f"{dir_path_stats}/stats-mean_by_level.nc", "rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
# 再次类似于第4行，这行代码打开了第三个文件stats-stddev_by_level.nc。
with open(f"{dir_path_stats}/stats-stddev_by_level.nc", "rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()

In [12]:
diffs_stddev_by_level

<xarray.Dataset>
Dimensions:            (level: 7)
Coordinates:
  * level              (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    so                 (level) float64 1.328e-14 -2.011e-14 ... -3.616e-15
    thetao             (level) float64 3.875e-15 2.13e-15 ... 1.861e-15
    siconc             float64 7.027e-16
    sithick            float64 -1.435e-15
    zos                float64 -2.207e-16
    usi                float64 -9.675e-17
    vsi                float64 -8.981e-17
    year_progress_sin  float32 3.406e-08
    year_progress_cos  float32 -4.154e-05
    day_progress_sin   float32 4.037e-09
    day_progress_cos   float32 -6.055e-09

In [13]:
mean_by_level

<xarray.Dataset>
Dimensions:            (level: 7)
Coordinates:
  * level              (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    so                 (level) float64 33.23 33.24 33.25 33.26 33.27 33.29 33.33
    thetao             (level) float64 13.65 13.64 13.63 13.63 13.62 13.62 13.62
    siconc             float64 0.6987
    sithick            float64 1.053
    zos                float64 -0.135
    usi                float64 0.000134
    vsi                float64 0.006718
    year_progress_sin  float32 0.05155
    year_progress_cos  float32 0.9981
    day_progress_sin   float32 8.073e-09
    day_progress_cos   float32 6.661e-08

In [14]:
stddev_by_level

<xarray.Dataset>
Dimensions:            (level: 7)
Coordinates:
  * level              (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    so                 (level) float64 5.71 5.703 5.695 5.685 5.676 5.667 5.623
    thetao             (level) float64 11.86 11.85 11.85 11.84 11.84 11.83 11.83
    siconc             float64 0.3718
    sithick            float64 0.7314
    zos                float64 0.7201
    usi                float64 0.09613
    vsi                float64 0.09308
    year_progress_sin  float32 0.03435
    year_progress_cos  float32 0.001847
    day_progress_sin   float32 0.7071
    day_progress_cos   float32 0.7071

In [15]:
# 检查纬度属性
print("Original Latitude Attributes:", example_batch['lat'].attrs)

# 修改属性
example_batch['lat'].attrs['valid_min'] = -90.0
example_batch['lat'].attrs['valid_max'] = 90.0

# 检查修改后的属性
print("Updated Latitude Attributes:", example_batch['lat'].attrs)

Original Latitude Attributes: {'axis': 'Y', 'long_name': 'Latitude', 'standard_name': 'latitude', 'step': 0.08333587646484375, 'unit_long': 'Degrees North', 'units': 'degrees_north', 'valid_max': 90.0, 'valid_min': -80.0}
Updated Latitude Attributes: {'axis': 'Y', 'long_name': 'Latitude', 'standard_name': 'latitude', 'step': 0.08333587646484375, 'unit_long': 'Degrees North', 'units': 'degrees_north', 'valid_max': 90.0, 'valid_min': -90.0}


In [16]:
# @title 选择绘图数据

# 这行代码创建了一个下拉菜单控件plot_example_variable，允许用户从数据集中的变量中选择一个来绘制。options参数使用example_batch.data_vars.keys()列出了所有可用的数据变量。
# 默认值value设置为"2m_temperature"，即2米温度。description参数提供了控件的标签，这里是“变量”。
plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="siconc",
    description="变量")
# 这行代码创建了一个下拉菜单控件plot_example_level，允许用户选择一个特定的气压级别来绘制数据。options参数使用example_batch.coords["level"].values列出了所有可用的气压级别。
# 默认值value设置为500，表示500百帕斯卡。description参数提供了控件的标签，这里是“级别”。
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=np.float32(0.494025),
    description="深度")
# 这行代码创建了一个复选框控件plot_example_robust，允许用户选择是否使用鲁棒性绘图。
# 鲁棒性绘图通常会忽略异常值，使得颜色映射更集中于数据的主要部分。默认值value设置为True，表示默认启用鲁棒性绘图。description参数提供了控件的标签，这里是“鲁棒性”。
plot_example_robust = widgets.Checkbox(value=True, description="鲁棒性")
# 创建了一个整数滑块控件plot_example_max_steps，允许用户选择绘图时使用的最大时间步数。min参数设置了滑块的最小值为1，max参数使用example_batch.dims["time"]设置了滑块的最大值为数据集的时间维度大小。
# 默认值value也设置为数据集的时间维度大小，表示默认使用所有时间步。description参数提供了控件的标签，这里是“最大步”。
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],
    description="最大步")

# 这段代码创建了一个垂直盒子布局控件VBox，包含了上述所有控件和一个标签。标签提醒用户运行下一个单元格以绘制数据，并告知如果重新运行此单元格，将清除他们的选择。
widgets.VBox([
    plot_example_variable,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="运行下一个单元格以绘制数据。重新运行此单元格将清除您的选择。")
])

In [17]:
# @title 绘制示例数据

# 这行代码设置了绘图的大小，将其作为一个变量plot_size。
plot_size = 7

# 这里创建了一个字典data，其中包含一个键值对。键是一个空格字符串，值是通过调用select函数和scale函数处理后的数据。
# select函数从example_batch中选择了特定的变量、级别和时间步数，然后scale函数对数据进行了缩放。
data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
              robust=plot_example_robust.value),
}
# 这段代码设置了图形的标题fig_title。如果数据集中的变量包含一个名为“等级”的坐标，那么标题将包含级别信息。
fig_title = plot_example_variable.value
if "等级" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

# # 调用了plot_data函数，将数据、标题、绘图大小和鲁棒性作为参数传递给它，以绘制数据。
plot_data(data, fig_title, plot_size, plot_example_robust.value)


In [18]:
# @title 选择要提取的训练和评估数据

# 创建了一个整数滑块（IntSlider），用于选择训练步数
train_steps = widgets.IntSlider(
#     设置滑块的初始值为1
    value=12, min=1, max=example_batch.sizes["time"]-2, description="训练步数")
eval_steps = widgets.IntSlider(
    value=example_batch.sizes["time"]-2, min=1, max=example_batch.sizes["time"]-2, description="评估步数")

widgets.VBox([
    train_steps,
    eval_steps,
    widgets.Label(value="运行下一个单元格以提取数据。重新运行此单元格将清除您的选择。")
])

In [19]:
# @title 提取训练和评估数据

print("数据集的维度:", example_batch.dims)

# 打印数据集的坐标
print("数据集的坐标:", example_batch.coords)

# 调用了一个名为 data_utils.extract_inputs_targets_forcings 的函数，并将其返回的结果分配给三个变量：train_inputs、train_targets 和 train_forcings。
train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
#     example_batch 是一个示例数据批次，target_lead_times 是一个时间切片，用于选择目标数据的时间范围。在这里，我们选择了从24小时到训练步数乘以24小时的时间范围。
    example_batch, target_lead_times=slice("24h", f"{train_steps.value*24}h"),
#     这是函数的第二个参数，它使用 dataclasses.asdict 将 task_config 转换为字典，并将其作为关键字参数传递给函数。
    **dataclasses.asdict(task_config))

# 这行代码类似于第1行，但是用于提取评估数据而不是训练数据。
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("24h", f"{eval_steps.value*24}h"),
    **dataclasses.asdict(task_config))

print("所有示例：  ", example_batch.dims.mapping)
print("训练输入：  ", train_inputs.dims.mapping)
print("训练目标： ", train_targets.dims.mapping)
print("训练强迫：", train_forcings.dims.mapping)
print("评估输入：   ", eval_inputs.dims.mapping)
print("评估目标：  ", eval_targets.dims.mapping)
print("评估强迫项: ", eval_forcings.dims.mapping)

train_inputs
# train_targets 
# train_forcings
eval_targets
eval_inputs
eval_forcings
# inputs和forcings有year_progress

数据集的维度: Frozen({'batch': 9, 'time': 7, 'lat': 539, 'lon': 1080, 'level': 7})
数据集的坐标: Coordinates:
  * level     (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon       (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat       (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 5 days 6 days
  * batch     (batch) int64 0 1 2 3 4 5 6 7 8
    datetime  (batch, time) datetime64[ns] 2021-04-30 2021-05-01 ... NaT
原始数据集维度: Frozen({'batch': 9, 'time': 7, 'lat': 539, 'lon': 1080, 'level': 7})
选择特定level层级后的数据集维度: Frozen({'batch': 9, 'time': 7, 'lat': 539, 'lon': 1080, 'level': 7})
{2}
{2}
{3}
{3}
提取时间后的输入数据集维度: Frozen({'batch': 9, 'time': 2, 'lat': 539, 'lon': 1080, 'level': 7})
提取时间后的目标数据集维度: Frozen({'batch': 9, 'time': 5, 'lat': 539, 'lon': 1080, 'level': 7})
最终的输入数据集维度: Frozen({'batch': 9, 'time': 2, 'level': 7, 'lat': 539, 'lon': 1080})
最终的目标数据集维度: Frozen({'batch': 9, 'time': 5, 'l

<xarray.Dataset>
Dimensions:            (batch: 9, time: 5, lat: 539, lon: 1080)
Coordinates:
  * lon                (lon) float32 -179.9 -179.5 -179.2 ... 179.1 179.5 179.8
  * lat                (lat) float64 -89.83 -89.5 -89.17 ... 88.83 89.17 89.5
  * time               (time) timedelta64[ns] 1 days 2 days 3 days 4 days 5 days
  * batch              (batch) int64 0 1 2 3 4 5 6 7 8
Data variables:
    zos                (batch, time, lat, lon) float64 nan nan nan ... 0.0 0.0
    vsi                (batch, time, lat, lon) float64 nan nan nan ... 0.0 0.0
    usi                (batch, time, lat, lon) float64 nan nan nan ... 0.0 0.0
    year_progress_sin  (batch, time) float32 0.8669 0.8582 ... 0.03393 0.6155
    year_progress_cos  (batch, time) float32 -0.4985 -0.5134 ... -0.9994 -0.7881
    day_progress_sin   (batch, time, lon) float32 -0.002182 ... -0.7959
    day_progress_cos   (batch, time, lon) float32 -1.0 -1.0 ... 0.6008 0.6054

In [20]:
# 定义构建和包装GraphCast预测器的函数
def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # 创建一个更深层次的一步预测器
  predictor = graphcast.GraphCast(model_config, task_config)

  # 修改输入/输出以处理从float32到BFloat16的转换
  predictor = casting.Bfloat16Cast(predictor)

  # 在应用输入/目标的规范化之后，进行BFloat16的转换
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # 包装所有内容，使一步模型能够产生轨迹
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor

# 定义前向运算函数
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

# 定义计算损失函数的函数
@hk.transform_with_state
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

# 定义计算梯度的函数
def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), model_config, task_config,
        i, t, f)
    return loss, (diagnostics, next_state)
  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads

# 定义一个函数，用于通过functools.partial传递配置
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# 定义一个函数，用于通过functools.partial传递参数和状态
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# 定义一个函数，用于丢弃状态并只返回预测结果
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

# 使用jax.jit编译初始化函数
init_jitted = jax.jit(with_configs(run_forward.init))

# 如果参数为空，则初始化参数和状态
if params is None:
  print(train_inputs.dims)
  print(train_inputs.coords)
  print(train_targets.dims)
  print(train_targets.coords)
  params, state = init_jitted(
      rng=jax.random.PRNGKey(0),
      inputs=train_inputs,
      targets_template=train_targets,
      forcings=train_forcings)

# 编译损失函数和梯度函数
loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
    run_forward.apply))))


Frozen({'batch': 9, 'time': 2, 'level': 7, 'lat': 539, 'lon': 1080})
Coordinates:
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time     (time) timedelta64[ns] -1 days 00:00:00
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
Frozen({'batch': 9, 'time': 5, 'level': 7, 'lat': 539, 'lon': 1080})
Coordinates:
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time     (time) timedelta64[ns] 1 days 2 days 3 days 4 days 5 days
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
stacked_array.sizes: Frozen({'batch': 9, 'lat': 539, 'lon': 1080, 'channels': 16})
template_dataset.sizes: Frozen({'batch': 9, 'time': 1, 'lat': 539, 'lon': 1080, 'leve

In [21]:
# print("params:", params)
print("state", state)

state {}


# 运行模型

请注意，第一次运行下面的单元格可能需要一段时间（可能几分钟），因为这包括代码编译的时间。第二次运行时速度会明显加快。

这将使用 python 循环迭代预测步骤，其中 1 步的预测是固定的。这比下面的训练步骤对内存的要求要低，应该可以使用小型 GraphCast 模型对 1 度分辨率数据进行 4 步预测。

In [22]:
# 确保模型分辨率与数据分辨率匹配
assert model_config.resolution in (0, 360. / eval_inputs.sizes["lon"]), (
  "Model resolution doesn't match the data resolution. You likely want to "
  "re-filter the dataset list, and download the correct data.")

# 打印输入、目标和强迫项的维度映射
print("Inputs:  ", eval_inputs.dims.mapping)
print("Targets: ", eval_targets.dims.mapping)
print("Forcings:", eval_forcings.dims.mapping)

# 使用chunked_prediction函数进行模型预测
predictions = rollout.chunked_prediction(
    run_forward_jitted,  # 编译过的前向运算函数
    rng=jax.random.PRNGKey(0),  # 随机数生成器的种子
    inputs=eval_inputs,  # 评估输入
    targets_template=eval_targets,  # 评估目标，乘以np.nan生成NaN值
    # targets_template=eval_targets * np.nan,  # 评估目标，乘以np.nan生成NaN值
    forcings=eval_forcings)  # 评估强迫项
predictions  # 输出预测结果

Inputs:   {'batch': 9, 'time': 2, 'level': 7, 'lat': 539, 'lon': 1080}
Targets:  {'batch': 9, 'time': 5, 'level': 7, 'lat': 539, 'lon': 1080}
Forcings: {'batch': 9, 'time': 5, 'lat': 539, 'lon': 1080}


stacked_array.sizes: Frozen({'batch': 9, 'lat': 539, 'lon': 1080, 'channels': 16})
template_dataset.sizes: Frozen({'batch': 9, 'time': 1, 'lat': 539, 'lon': 1080, 'level': 7})
stacked_array: <xarray.Variable (batch: 9, lat: 539, lon: 1080, channels: 16)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[9,539,1080,16])>with<DynamicJaxprTrace(level=2/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (batch: 9, time: 1, lat: 539, lon: 1080, level: 7)
Coordinates:
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time     (time) timedelta64[ns] 1 days
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
Data variables:
    siconc   (batch, time, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    sithick  (batch, time, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    so       (batch, time, level,

<xarray.Dataset>
Dimensions:  (time: 5, batch: 9, lat: 539, lon: 1080, level: 7)
Coordinates:
  * time     (time) timedelta64[ns] 1 days 2 days 3 days 4 days 5 days
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
Data variables:
    siconc   (time, batch, lat, lon) float32 nan nan nan nan ... nan nan nan nan
    sithick  (time, batch, lat, lon) float32 nan nan nan nan ... nan nan nan nan
    so       (time, batch, level, lat, lon) float32 nan nan nan ... nan nan nan
    thetao   (time, batch, level, lat, lon) float32 nan nan nan ... nan nan nan

In [1]:
eval_targets

NameError: name 'eval_targets' is not defined

In [23]:
# @title 选择要绘制的预测结果

# 假设predictions是你的xarray Dataset对象
print(predictions.coords.keys())  # 打印所有坐标键

plot_pred_variable = widgets.Dropdown(
    options=predictions.data_vars.keys(),
    value="siconc",
    description="变量")
# plot_pred_level = widgets.Dropdown(
#     options=predictions.coords["depth"].values,
#     value=0.494025,
#     description="深度")
plot_pred_robust = widgets.Checkbox(value=True, description="鲁棒性")
plot_pred_max_steps = widgets.IntSlider(
    min=1,
    max=predictions.dims["time"],
    value=predictions.dims["time"],
    description="最大步")

widgets.VBox([
    plot_pred_variable,
    # plot_pred_level,
    plot_pred_robust,
    plot_pred_max_steps,
    widgets.Label(value="运行下一个单元格，绘制预测结果。重新运行该单元格将清除您的选择。")
])

KeysView(Coordinates:
  * time     (time) timedelta64[ns] 1 days 2 days 3 days 4 days 5 days
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8)


In [24]:
# 设置绘图标题
# @title 使用预测数据绘图

# 设置绘图大小
plot_size = 5

# 计算最大步数，用于绘图
plot_max_steps = min(predictions.dims["time"], plot_pred_max_steps.value)

# 准备绘图数据
data = {
    # 缩放目标数据 
    # "Inputs": scale(select(eval_inputs, plot_pred_variable.value,  plot_max_steps), robust=plot_pred_robust.value),
    "Targets": scale(select(eval_targets, plot_pred_variable.value,  plot_max_steps), robust=plot_pred_robust.value),
    # 缩放预测数据
    "Predictions": scale(select(predictions, plot_pred_variable.value, plot_max_steps), robust=plot_pred_robust.value),
    # 计算并缩放目标与预测的差异
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_max_steps) -
                        select(predictions, plot_pred_variable.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}

# 设置图表标题
fig_title = plot_pred_variable.value
# 如果预测数据中包含“level”坐标，则在标题中添加相应的气压层级
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

# 调用plot_data函数进行绘图
plot_data(data, fig_title, plot_size, plot_pred_robust.value)


/root/miniconda3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/root/miniconda3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


In [ ]:
residual = eval_targets - last_input  # target 为真实值，last_input 为输入的最后一个时间步

# 训练模型

以下操作需要大量内存，而且根据所使用的加速器，只能在低分辨率数据上拟合很小的 "随机 "模型。它使用上面选择的训练步数。

第一次执行单元需要更多时间，因为其中包括函数的 jit 时间。

In [25]:
# 设置标题，描述代码块的功能
# @title 损失计算（多步骤递归（自回归）损失）

# 调用已经编译过的损失函数，计算损失值和诊断信息
loss, diagnostics = loss_fn_jitted(
    rng=jax.random.PRNGKey(0),  # 使用随机数生成器的种子
    inputs=train_inputs,        # 训练输入
    targets=train_targets,      # 训练目标
    forcings=train_forcings)    # 训练强迫项

# 打印损失值
print("Loss:", float(loss))

stacked_array.sizes: Frozen({'batch': 9, 'lat': 539, 'lon': 1080, 'channels': 16})
template_dataset.sizes: Frozen({'time': 1, 'batch': 9, 'lat': 539, 'lon': 1080, 'level': 7})
stacked_array: <xarray.Variable (batch: 9, lat: 539, lon: 1080, channels: 16)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[9,539,1080,16])>with<DynamicJaxprTrace(level=3/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (time: 1, batch: 9, lat: 539, lon: 1080, level: 7)
Coordinates:
  * time     (time) timedelta64[ns] 1 days
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    siconc   (time, batch, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    sithick  (time, batch, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    so       (time, batch, level,

In [26]:
# 设置标题，描述代码块的功能
# @title 梯度计算（通过时间进行反推）

# 调用已经编译过的梯度函数，计算梯度值
loss, diagnostics, next_state, grads = grads_fn_jitted(
    inputs=train_inputs,        # 训练输入
    targets=train_targets,      # 训练目标
    forcings=train_forcings)    # 训练强迫项

# 计算梯度的平均绝对值
# 首先，使用tree_map函数将梯度中的每个元素取绝对值并计算平均值，然后使用tree_flatten函数将结果展平并计算平均值。
mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])

# 打印损失值和平均梯度绝对值
# {loss:.4f}和{mean_grad:.6f}是格式化字符串，用于将损失值和平均梯度绝对值显示为指定的小数位数。
print(f"Loss: {loss:.4f}, Mean |grad|: {mean_grad:.6f}")


stacked_array.sizes: Frozen({'batch': 9, 'lat': 539, 'lon': 1080, 'channels': 16})
template_dataset.sizes: Frozen({'time': 1, 'batch': 9, 'lat': 539, 'lon': 1080, 'level': 7})
stacked_array: <xarray.Variable (batch: 9, lat: 539, lon: 1080, channels: 16)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[9,539,1080,16])>with<DynamicJaxprTrace(level=5/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (time: 1, batch: 9, lat: 539, lon: 1080, level: 7)
Coordinates:
  * time     (time) timedelta64[ns] 1 days
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
Data variables:
    siconc   (time, batch, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    sithick  (time, batch, lat, lon) bfloat16 xarray_jax.JaxArrayWrapper(Trac...
    so       (time, batch, level,

2024-08-07 15:25:50.723492: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 0: 3.32595e+06, expected 3.94854e+06
2024-08-07 15:25:50.723518: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 1: 3.32595e+06, expected 3.94854e+06
2024-08-07 15:25:50.723521: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 2: 3.34234e+06, expected 3.96493e+06
2024-08-07 15:25:50.723524: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 3: 3.32595e+06, expected 3.94854e+06
2024-08-07 15:25:50.723526: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 4: 3.34234e+06, expected 3.96493e+06
2024-08-07 15:25:50.723546: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 5: 3.34234e+06, expected 3.98131e+06
2024-08-07 15:25:50.723549: E external/xla/xla/service/gpu/buffer_comparator.cc:152] Difference at 6: 3.34234e+06, expected 3.96493e+06
2024-08-07 15:25:50.723551: E external/xla/xla/s

Loss: nan, Mean |grad|: nan


In [27]:
# 设置标题，描述代码块的功能
# @title 递归（自回归）推出（在 JAX 中保持循环）
# 使用训练数据进行模型的自回归预测，并输出预测结果

# 打印输入、目标和强迫项的维度映射
# 打印train_inputs的维度映射，显示训练输入数据的维度信息。
print("Inputs:  ", train_inputs.dims.mapping)
# 打印train_targets的维度映射，显示训练目标数据的维度信息
print("Targets: ", train_targets.dims.mapping)
# 打印train_forcings的维度映射，显示训练强迫项数据的维度信息
print("Forcings:", train_forcings.dims.mapping)

# 使用编译过的前向运算函数进行预测
predictions = run_forward_jitted(
    rng=jax.random.PRNGKey(0),  # 随机数生成器的种子
    inputs=train_inputs,        # 训练输入
    targets_template=train_targets * np.nan,  # 创建一个包含NaN值的目标模板，targets_template参数创建了一个目标模板，其中的值都是NaN（不是数字）。这通常用于初始化目标数组，在自回归预测中填充预测值。
    forcings=train_forcings)    # 训练强迫项
predictions  # 输出预测结果


Inputs:   {'batch': 9, 'time': 2, 'level': 7, 'lat': 539, 'lon': 1080}
Targets:  {'batch': 9, 'time': 5, 'level': 7, 'lat': 539, 'lon': 1080}
Forcings: {'batch': 9, 'time': 5, 'lat': 539, 'lon': 1080}
stacked_array.sizes: Frozen({'batch': 9, 'lat': 539, 'lon': 1080, 'channels': 16})
template_dataset.sizes: Frozen({'batch': 9, 'time': 1, 'lat': 539, 'lon': 1080, 'level': 7})
stacked_array: <xarray.Variable (batch: 9, lat: 539, lon: 1080, channels: 16)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(bfloat16[9,539,1080,16])>with<DynamicJaxprTrace(level=3/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (batch: 9, time: 1, lat: 539, lon: 1080, level: 7)
Coordinates:
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * time     (time) timedelta64[ns] 1 days
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
Data varia

<xarray.Dataset>
Dimensions:  (time: 5, batch: 9, lat: 539, lon: 1080, level: 7)
Coordinates:
  * time     (time) timedelta64[ns] 1 days 2 days 3 days 4 days 5 days
  * level    (level) float32 0.494 1.541 2.646 3.819 5.078 6.441 7.93
  * lon      (lon) float32 -179.9 -179.5 -179.2 -178.9 ... 179.1 179.5 179.8
  * lat      (lat) float64 -89.83 -89.5 -89.17 -88.83 ... 88.5 88.83 89.17 89.5
  * batch    (batch) int64 0 1 2 3 4 5 6 7 8
Data variables:
    siconc   (time, batch, lat, lon) float32 xarray_jax.JaxArrayWrapper(Array...
    sithick  (time, batch, lat, lon) float32 xarray_jax.JaxArrayWrapper(Array...
    so       (time, batch, level, lat, lon) float32 xarray_jax.JaxArrayWrappe...
    thetao   (time, batch, level, lat, lon) float32 xarray_jax.JaxArrayWrappe...